# PDB preparation for alpha-galactosidase A

This notebook prepares apo and holo systems of α-galactosidase A for MD simulations (untill equilibration).

Remember to activate the conda environment with ```conda activate htmd```.
If the required software is not installed yet, please read the README file in this repository.




### Sections
First, run the following cells to initiate the correct apo and DGJ mol objects.

Then:  
[APO only build](#apo)  
[DGJ only build](#holo-(DGJ))  

In [ ]:
#IMPORT PACKAGES
from htmd.ui import *
import pandas as pd 
import numpy as np
import os 
from htmd.builder import charmm
from acemd.protocols import setup_equilibration
import re 


The following cell contains:
- PART 1: modifiable data;
- PART 2: fixed patches for the glycans, glycosylation sites and ligand import files;
- PART 3: Molecule preparation and cleanup.


In [ ]:
###################### PART 1 ######################
###################### CHANGE DATA AS PLEASED 

parent_folder='prepared_systems' #main folder different PDB-generated systems are stored
molecule_folder = '3s5y' #parent folder for organising wt/mut and apo/holo
folder = f'{parent_folder}/{molecule_folder}' #final case subfolder

molecule = '../glycosylation/3s5y_reglyco.pdb' #PDB path
#the PDB is obtained from rcsb.org and it is expected to be modelled with glycoshape.org

#IF NOTHING TO REMOVE, LEAVE THE LISTS EMPTY.
mutations = [('resid 215', 'SER')] #resid to mutate. #('resid 215', 'SER'),('resid 301', 'GLN')
#if no mutations, wt systems will be prepared

resnames_to_remove = []  #'SO4','HOH', 'NOJ'
#NOTE if the molecule is previously processed by glycoshape, water and ions should be already removed

# EQUILIBRATION AND SIMULATION DATA
eq_run = '50 ns' #also in us, ns, ps and fs
eq_temp = 300
minimize = 1000
prod_run = '1 us' #also in us, ns, ps and fs
prod_temp = 300



###################### PART 2 ######################
###################### DO NOT CHANGE FROM HERE

#default glycan patches
glycan_patch = ['patch 14BB P2:2 P2:3', 'patch 14BB P3:2 P3:3', 'patch 14BB P4:2 P4:3', 'patch 14BB P5:2 P5:3', 'patch 14BB P6:2 P6:3', 'patch 14BB P7:2 P7:3',
         'patch 14BB P2:3 P2:4', 'patch 14BB P3:3 P3:4', 'patch 14BB P4:3 P4:4', 'patch 14BB P5:3 P5:4', 'patch 14BB P6:3 P6:4', 'patch 14BB P7:3 P7:4',
         'patch 13AB P2:4 P2:6', 'patch 13AB P3:4 P3:6', 'patch 13AB P4:4 P4:6', 'patch 13AB P5:4 P5:6', 'patch 13AB P6:4 P6:6', 'patch 13AB P7:4 P7:6',
         'patch 16AT P2:4 P2:5', 'patch 16AT P3:4 P3:5', 'patch 16AT P4:4 P4:5', 'patch 16AT P5:4 P5:5', 'patch 16AT P6:4 P6:5', 'patch 16AT P7:4 P7:5',
         ] #top/top_all36_carb.rtf
#we did not included here the glycosylation sites patches as they can change position with different pdbs.
#glycosylation patches are of form "patch NGLB <protein segid>:<protein resid> <glycan segid>:<glycan resid>"
#assuming the glycans are generated via glycoshape.org (ID:GM0026MO), we set the glycan patches as fixed and assume that the first residue of each glycan has id 2 

#glycosylation site
gly_resid = [139, 192, 215]
#the three glycosilation sites have been evaluated in literature

#correct ligand for Holo str
#to model alternative ligands, replace these files with the corresponding
#CGenFF-parameterized structures for chain A and chain B, keeping the rest
#of the protocol unchanged.
dgj_a = '../DGJ/3s5y/DGJ_A.mol2' #chain A 
dgj_b = '../DGJ/3s5y/DGJ_B.mol2' #chain B

#MAKE FOLDER FOR {MOLECULE}
os.makedirs(f'../{folder}', exist_ok=True) #check and make

#AMINO ACID CODES FOR CONVERSION (do not change)
aa_3to1 = {'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F', 'GLY': 'G', 
 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L', 'MET': 'M', 'ASN': 'N', 
 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R', 'SER': 'S', 'THR': 'T', 'VAL': 'V', 
 'TRP': 'W', 'TYR': 'Y'} #conversion from 3-letter code to 1-letter code

print(f'Protein structure: {molecule_folder}, mutated residues: {mutations}, glycosilated residues: {gly_resid}')


###################### PART 3 ######################
###################### COMMON MOLECULE PREPARATION PART 

#read the molecule pdb, remove the unwanted molecules 
mol = Molecule(molecule)

#REMOVE RESNAMES
print(f'\nRemoving residues: {resnames_to_remove}') #check
original_count = mol.numAtoms
if resnames_to_remove:
    selection = ' or '.join([f'resname {res}' for res in resnames_to_remove]) #build correct selection
    mol.filter(f'not ({selection})') #remove
removed_count = original_count - mol.numAtoms #check
print(f'Removed {removed_count} atoms')
print(f'Remaining atoms: {mol.numAtoms}')

#CREATE DUPLICATE FOR MOL_APO 
mol_apo = mol.copy()
mol_DGJ = mol.copy()

print('Systems are ready to be prepared.')

From this point on apo and holo are run from different cells:

[APO only build](#apo)  
[DGJ only build](#holo-(DGJ))  

## APO
[Back to main](#sections) 

In [ ]:
#HANDLE MUTATIONS
proteins_apo=[]

if mutations: #<resid  resid> and <mut resname>
    for mutation in mutations:
        resid_sel, new_res = mutation 
        wt = np.unique(mol_apo.get('resname', resid_sel))
        wt = aa_3to1[wt[0]] 
        num = resid_sel.split(' ')[1]
        mut = aa_3to1[new_res]

        folder_apo = f'../{folder}/apo_{wt}{num}{mut}'
        os.makedirs(folder_apo, exist_ok=True)
        print(f'Storing generated data at {folder_apo}.')
        
        #MAKE A COPY OF THE APO TO BE MUTATED
        print(f'Mutating protein at {mutation}')
        mol_apo_mut = mol_apo.copy()
        mol_apo_mut.mutateResidue(resid_sel, new_res)
            
        proteins_apo.append((f'apo_{wt}{num}{mut}', mol_apo_mut, folder_apo, num)) 

#FOR WILD TYPE
else: #no mutations
    print('No mutation specified: preparing a wild-type system.')
    folder_apo = f'../{folder}/apo'
    os.makedirs(folder_apo, exist_ok=True) 
    proteins_apo.append(('apo', mol_apo, folder_apo, None)) #tolto colonna segid to remove 



#COMMON
for label, mol, folder_apo, num in proteins_apo:
    segid_to_remove = [] #in case of mut 
    print(f"\nProcessing system: {label}")

    #SYSTEM PREPARATION AND SEGMENTATION.
    system_apo, data = systemPrepare(mol, pH=7.0, return_details=True, plot_pka=f'{folder_apo}/{molecule_folder}_{label}_pka')
    system_apo = autoSegment(system_apo) #segment system
    
#N-GLYCO PATCHES GENERATION
    prot = np.unique(system_apo.get('segid', 'protein'))  # protein segment IDs
    glyc = np.unique(system_apo.get('segid', 'resname NAG'))  # glycan segment IDs

    patch_data = {}
    for p in prot:
        for r in gly_resid:
            for g in glyc:
                #being so strict selection-wise if there is no ASN no patch is generated
                pair = system_apo.atomselect(f'(segid {g} and resname NAG and name C1) and within 5 of (segid {p} and resname ASN and resid {r} and name ND2)',indexes=True)
                patch_data[(p, r, g)] = pair 
                if pair:
                    print('Found N-linkage between:', p, r, g)
    filtered = {k: v for k, v in patch_data.items() if v.size > 0} #remove the non matches

    #in case of mutation of the glycosilation site find the segid to be removed
    glyc_in_patches = {g for (_, _, g) in filtered.keys()}
    glyc_not_patched = [g for g in glyc if g not in glyc_in_patches]

    print('REMOVING THE FOLLOWING GLYCANS, THE GLYCOSILATION SITE IS MUTATED:', glyc_not_patched)
    #print(glyc)

    patch_list = []
    for (p, r, g), pair in filtered.items():
        patch_str = f"patch NGLB {p}:{r} {g}:2" #protein - glycan patch (the others are fixed) 
        patch_list.append(patch_str)

    patch = patch_list + glycan_patch #as required by htmd

    print('ALL PATCHES:', patch)

    #patch removal in case of mutation on the glycosylated site
    if num is not None: #col non empty, not wt
        if int(num) in gly_resid:
            print(f'MUTATION IN GLYCOSILATION SITE:{num}')
            glyc_not_patched = list(glyc_not_patched)   
            segid_to_remove.extend(glyc_not_patched)       
            new_patch=[]
            for p in patch:
                parts = p.split()
                segids_in_patch = [part.split(':')[0] for part in parts[2:]]
                
                if any(s in glyc_not_patched for s in segids_in_patch):
                    print(f'REMOVING PATCH (glycan removed): {p}')
                    continue
                new_patch.append(p)
            patch = new_patch
 
    #remove segid of glycans if in mutation site
    if segid_to_remove:
        sel = ' or '.join([f'segid {sgd}' for sgd in segid_to_remove])
        system_apo.filter(f'not ({sel})')
        print(f"REMOVED GLYCANS AT: {sel}")

    #intermediate saving (non mandatory)
    data.to_csv(f'{folder_apo}/{molecule_folder}_{label}_prep.csv') #save
    system_apo.write(f'{folder_apo}/{molecule_folder}_{label}_prep.pdb') #save

    segments_apo = np.unique(system_apo.segid) #check
    print(f'Segments: {segments_apo}')

    #SYSTEM SOLVATION
    system_solv_apo = solvate(system_apo, negx = 20  , negy = 20, negz = 20, posx = 20, posy = 20, posz = 20)
    system_solv_apo.write(f'{folder_apo}/{molecule_folder}_{label}_solv.pdb')
    #system_solv.write(f'{folder_apo}/{molecule}_solv.psf') #non giusto

    #SYSTEM BUILDING WITH CHARMM36m AND PATCHES
    #other parameters available
    system_charmm_apo = charmm.build(system_solv_apo,  saltconc = 0.15, saltanion = 'CL', saltcation = 'K',
                                topo= ['top/top_all36_prot.rtf', 'top/top_all36_carb.rtf', 'top/top_water_ions.rtf', 'top/top_all36_cgenff.rtf', '../DGJ/top_DGJ.rtf'],    
                                param=['par/par_all36m_prot.prm','par/par_all36_carb.prm','par/par_water_ions.prm', 'par/par_all36_cgenff.prm', '../DGJ/par_DGJ.prm'],
                                stream=['str/carb/toppar_all36_carb_glycopeptide.str'],
                                patches = patch,  
                                outdir = f'{folder_apo}/build') #patches = patch,
    print('build/ folder generated.') 

    #MINIMIZATION AND EQUILIBRATION PREPARATION
    setup_equilibration(builddir=f'{folder_apo}/build', 
                        outdir=f'{folder_apo}/equilibration',
                        run = eq_run, #also in us, ns, ps and fs
                        temperature = eq_temp,
                        coordinates = f'{folder_apo}/build/structure.pdb',
                        structure = f'{folder_apo}/build/structure.psf',
                        parameters = f'{folder_apo}/build/parameters.prm',
                        minimize = minimize)
    print('equilibration/ folder generated.') 

#to remember:
# "NA","MG","ZN","K","CS","CA","CL"  
#, 'noj/noj_g.rtf'
#charmm.listFiles()  #check for files  
# 


#### The **production** folder can be generated only **after the equilibration is compleded**.

In particular:

1. run equilibration
2. check equilibration ended with *check_end.py* 
3. run *production_prep.py*
4. run production

## HOLO (DGJ)
[Back to main](#sections) 

In [ ]:
#HANDLE MUTATIONS
proteins_DGJ=[]

if mutations: #<resid  resid> and <mut resname>
    for mutation in mutations:
        resid_sel, new_res = mutation 
        wt = np.unique(mol_DGJ.get('resname', resid_sel))
        wt = aa_3to1[wt[0]] 
        num = resid_sel.split(' ')[1]
        mut = aa_3to1[new_res]

        folder_DGJ = f'../{folder}/DGJ_{wt}{num}{mut}'
        os.makedirs(folder_DGJ, exist_ok=True)
        print(f'Storing generated data at {folder_DGJ}.')
        
        #MAKE A COPY OF THE APO TO BE MUTATED
        print(f'Mutating protein at {mutation}')
        mol_DGJ_mut = mol_DGJ.copy()
        mol_DGJ_mut.mutateResidue(resid_sel, new_res)
           
        proteins_DGJ.append((f'DGJ_{wt}{num}{mut}', mol_DGJ_mut, folder_DGJ, num))  

#FOR WILD TYPE
else: #no mutations
    print('No mutation specified: preparing a wild-type system.')
    folder_DGJ = f'../{folder}/DGJ'
    os.makedirs(folder_DGJ, exist_ok=True) 
    proteins_DGJ.append(('DGJ', mol_DGJ, folder_DGJ, None)) #tolto colonna segid to remove 



#COMMON
for label, mol, folder_DGJ, num in proteins_DGJ: 
    segid_to_remove = [] # in case of mut
    print(f"\nProcessing system: {label}")

    #APPEND CORRECT DGJ IN CHAIN A AND B
    #chain A
    DGJ_A = Molecule(dgj_a)
    DGJ_A.set('resid', '1', 'resname DGJ')
    DGJ_A.set('chain', 'L', 'resname DGJ')
    DGJ_A.set('segid', 'P8', 'resname DGJ')
    
    #chain B
    DGJ_B = Molecule(dgj_b)
    DGJ_B.set('resid', '1', 'resname DGJ')
    DGJ_B.set('chain', 'M', 'resname DGJ')
    DGJ_B.set('segid', 'P9', 'resname DGJ')
    #append
    #mol.append(DGJ_A)
    #mol.append(DGJ_B)

    #SYSTEM PREPARATION AND SEGMENTATION.
    system_DGJ, data = systemPrepare(mol, pH=7.0, return_details=True, plot_pka=f'{folder_DGJ}/_{folder}_{label}_pka')
    system_DGJ = autoSegment(system_DGJ, basename='P') #segment system

    print('Appending one ligand (DGJ) in each monomer.')
    system_DGJ.append(DGJ_A)
    system_DGJ.append(DGJ_B)

#N-GLYCO PATCHES GENERATION
    prot = np.unique(system_DGJ.get('segid', 'protein'))  # protein segment IDs
    glyc = np.unique(system_DGJ.get('segid', 'resname NAG'))  # glycan segment IDs    

    patch_data = {}
    for p in prot:
        for r in gly_resid:
            for g in glyc:
                #being so strict selection-wise if there is no ASN no patch is generated
                pair = system_DGJ.atomselect(f'segid {g} and resname NAG and name C1 and within 5 of (segid {p} and resid {r} and name ND2)',indexes=True)
                patch_data[(p, r, g)] = pair
                if pair:
                    print('Found N-linkage between:', p, r, g)
    filtered = {k: v for k, v in patch_data.items() if v.size > 0} #remove non-matched cases


    #in case of mutation of the glycosilation site find the segid to be removed
    glyc_in_patches = {g for (_, _, g) in filtered.keys()}
    glyc_not_patched = [g for g in glyc if g not in glyc_in_patches]

    print('REMOVING THE FOLLOWING GLYCANS, THE GLYCOSILATION SITE IS MUTATED:', glyc_not_patched)
    #print(glyc)            

    patch_list = []
    for (p, r, g), pair in filtered.items():
        patch_str = f'patch NGLB {p}:{r} {g}:2' #protein - glycan patch (the others are fixed)
        patch_list.append(patch_str)

    patch = patch_list + glycan_patch 

    print('ALL PATCHES:', patch)
    
    #patch removal in case of mutation on the glycosylated site
    if num is not None: #col not empty, not wt
        if int(num) in gly_resid:
            print(f'MUTATION IN GLYCOSILATION SITE:{num}')
            glyc_not_patched = list(glyc_not_patched)   
            segid_to_remove.extend(glyc_not_patched)       
            new_patch=[]
            for p in patch:
                parts = p.split()
                segids_in_patch = [part.split(':')[0] for part in parts[2:]]
                
                if any(s in glyc_not_patched for s in segids_in_patch):
                    print(f'REMOVING PATCH (glycan removed): {p}')
                    continue
                new_patch.append(p)
            patch = new_patch


    #remove segid of glycans in mutation site
    if segid_to_remove:
        sel = ' or '.join([f'segid {sgd}' for sgd in segid_to_remove])
        system_DGJ.filter(f'not ({sel})')
        print(f"REMOVED GLYCANS AT: {sel}")

    #intermediate saving
    data.to_csv(f'{folder_DGJ}/{molecule_folder}_{label}_prep.csv') #save
    system_DGJ.write(f'{folder_DGJ}/{molecule_folder}_{label}_prep.pdb') #save

    segments_DGJ = np.unique(system_DGJ.segid) #check
    print(f'Final segments: {segments_DGJ}')

    #SYSTEM SOLVATION
    system_solv_DGJ = solvate(system_DGJ, negx = 20  , negy = 20, negz = 20, posx = 20, posy = 20, posz = 20)
    system_solv_DGJ.write(f'{folder_DGJ}/{molecule_folder}_{label}_solv.pdb')
    #system_solv.write(f'../{folder}/_solv.psf') #non giusto

    #SYSTEM BUILDING WITH CHARMM36m AND PATCHES
    #other parameters available
    system_charmm_DGJ = charmm.build(system_solv_DGJ,  saltconc = 0.15, saltanion = 'CL', saltcation = 'K',
                                topo= ['top/top_all36_prot.rtf', 'top/top_all36_carb.rtf', 'top/top_water_ions.rtf', 'top/top_all36_cgenff.rtf', '../DGJ/top_DGJ.rtf'],    
                                param=['par/par_all36m_prot.prm','par/par_all36_carb.prm','par/par_water_ions.prm', 'par/par_all36_cgenff.prm', '../DGJ/par_DGJ.prm'],
                                stream=['str/carb/toppar_all36_carb_glycopeptide.str'],
                                patches = patch,
                                outdir = f'{folder_DGJ}/build') 
    print('build/ folder generated.')                         

    #MINIMIZATION AND EQUILIBRATION PREPARATION
    setup_equilibration(builddir=f'{folder_DGJ}/build', 
                        outdir=f'{folder_DGJ}/equilibration',
                        run = eq_run, 
                        temperature = eq_temp,
                        coordinates = f'{folder_DGJ}/build/structure.pdb',
                        structure = f'{folder_DGJ}/build/structure.psf',
                        parameters = f'{folder_DGJ}/build/parameters.prm',
                        minimize = minimize)
    print('equilibration/ folder generated.')


    #to remember:
    # "NA","MG","ZN","K","CS","CA","CL'  
    #, 'noj/noj_g.rtf'
    #charmm.listFiles()  #check for files

#### The **production** folder can be generated only **after the equilibration is compleded**.

In particular:

1. run equilibration
2. check equilibration ended with *check_end.py* 
3. run *production_prep.py*
4. run production
 